# 第二篇配套 Demo：System Prompt 与 Instructions

这个 notebook 对应《上下文工程系列教程（二）》。

目标不是覆盖所有 API 细节，而是用几段尽量短小的代码，把文章里讲到的几个核心点跑通：

- `system prompt` 负责长期稳定规则
- `instructions` 负责当前任务要求
- 两者通常在一次请求里一起进入模型
- `instructions` 常常来自模板拼装，而不是直接等于用户原话


## 1. 依赖导入

这里先导入本 notebook 会用到的依赖：

- `os`：读取环境变量中的 API Key
- `json`：打印请求结构，方便观察 `system prompt` 和 `instructions` 是怎么组织的
- `textwrap`：格式化展示文本
- `openai.OpenAI`：演示 OpenAI 兼容客户端调用方式

如果本机还没有安装 `openai`，可以先运行：

```bash
pip install openai
```

In [ ]:
import json
import os
import textwrap

from openai import OpenAI

## 2. 基础配置

下面这个单元做两件事：

1. 从环境变量读取 API Key
2. 初始化一个客户端

为了让 demo 更容易迁移，这里把 `base_url` 也参数化了。

- 直接调用 OpenAI 时，`base_url` 可以留空
- 调用 DeepSeek、Ollama 或其他 OpenAI 兼容接口时，只需要替换 `base_url`


In [ ]:
API_KEY = os.getenv("OPENAI_API_KEY", "<YOUR_API_KEY>")
BASE_URL = os.getenv("OPENAI_BASE_URL")  # 例如 https://api.deepseek.com 或 http://localhost:11434/v1
MODEL = os.getenv("LLM_MODEL", "gpt-4.1-mini")

client_kwargs = {"api_key": API_KEY}
if BASE_URL:
    client_kwargs["base_url"] = BASE_URL

client = OpenAI(**client_kwargs)

print({"base_url": BASE_URL or "default", "model": MODEL})

## 3. 两个辅助函数

第一个函数用来打印请求结构，便于观察 `system prompt` 和 `instructions` 分别放在哪里。

第二个函数用来真正发请求。这里使用的是 OpenAI 兼容的 Chat Completions 形式，因为它最适合直观展示：

- `role=system`：长期规则层
- `role=user`：当前任务层


In [ ]:
def show_payload(system_prompt: str, instructions: str):
    payload = {
        "model": MODEL,
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": instructions},
        ],
    }
    print(json.dumps(payload, ensure_ascii=False, indent=2))


def run_chat_demo(system_prompt: str, instructions: str, temperature: float = 0.2):
    response = client.chat.completions.create(
        model=MODEL,
        temperature=temperature,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": instructions},
        ],
    )
    return response.choices[0].message.content

## 4. Demo 1：观察 System Prompt 的作用

这个 demo 固定任务内容，只改变 `system prompt`。

我们想观察的是：

- 当长期规则层变化时，模型整体行为会如何变化
- `system prompt` 更像“这个 Agent 应该怎么行动”，而不是“这次要做什么”


In [ ]:
task = "请分析下面的 Python 函数为什么会抛出 IndexError，并给出修复建议。不要直接修改代码。"

system_prompt_a = "你是一个谨慎的代码助手。先分析，再给出最小修改建议；不要直接重写整段代码。"
system_prompt_b = "你是一个追求快速产出的代码助手。优先直接给出修改后的代码。"

print("=== Payload A ===")
show_payload(system_prompt_a, task)

print("\n=== Payload B ===")
show_payload(system_prompt_b, task)

In [ ]:
print("=== Response A ===")
print(run_chat_demo(system_prompt_a, task))

print("\n=== Response B ===")
print(run_chat_demo(system_prompt_b, task))

这里可以重点观察两点：

- 任务本身没有变，但模型的行为风格会随着 `system prompt` 变化
- 这说明 `system prompt` 更适合承载长期规则，比如谨慎程度、输出习惯、工具使用原则


## 5. Demo 2：观察 Instructions 的作用

这次我们反过来：固定 `system prompt`，只改变当前任务 instructions。

我们想观察的是：

- 长期规则层不变时，任务要求如何驱动输出变化
- `instructions` 更像“这次具体要怎么做”


In [ ]:
shared_system_prompt = "你是一个谨慎的代码助手。先分析，再给出结论；保持回答简洁、工程化、可验证。"

instructions_a = "任务：解释这段代码为什么报错。输出格式：分成'原因'和'修复建议'两部分。"
instructions_b = "任务：解释这段代码为什么报错。输出格式：只给三条最关键结论，每条不超过20个字。"

print("=== Response A ===")
print(run_chat_demo(shared_system_prompt, instructions_a))

print("\n=== Response B ===")
print(run_chat_demo(shared_system_prompt, instructions_b))

这个 demo 适合对应文章里的一个核心判断：`System Prompt` 定义长期规则，`Instructions` 定义当前任务怎么执行。

同一个 system 层保持不变时，只要任务目标、输出格式、限制条件变化，模型输出也会相应变化。

## 6. Demo 3：用户输入不等于 Instructions

文章里提到，用户原话往往只是素材来源，不一定已经是一个可执行 instructions。

下面这个 demo 演示一个非常常见的做法：先把用户原话整理成结构化任务，再拼成 instructions。

In [ ]:
raw_user_input = "帮我看看这个报错，尽量少改代码，最好补个测试。"

structured_task = {
    "goal": "修复当前报错",
    "constraint": "尽量少改代码",
    "extra": "补充测试",
    "output": "说明修改原因与验证结果",
}

instruction_template = textwrap.dedent("""
任务：{goal}
限制：{constraint}
附加要求：{extra}
输出：{output}
""").strip()

final_instructions = instruction_template.format(**structured_task)

print("=== 用户原话 ===")
print(raw_user_input)

print("\n=== 结构化任务 ===")
print(json.dumps(structured_task, ensure_ascii=False, indent=2))

print("\n=== 最终 Instructions ===")
print(final_instructions)

## 7. Demo 4：模板化构造 Instructions

文章里还提到，`instructions` 最常见的工程实现方式，就是“模板 + 变量填充”。

下面这个例子演示一个简单的任务模板。真实系统里，变量来源可以是：

- 用户输入
- 上下文检索结果
- 当前运行状态
- 工具返回结果


In [ ]:
def build_instructions(task_goal, user_input, output_format, constraints):
    template = textwrap.dedent("""
    你需要完成以下任务：
    - 目标：{task_goal}
    - 用户提供材料：{user_input}
    - 输出格式：{output_format}
    - 限制条件：{constraints}
    """).strip()
    return template.format(
        task_goal=task_goal,
        user_input=user_input,
        output_format=output_format,
        constraints=constraints,
    )


demo_instructions = build_instructions(
    task_goal="分析代码报错原因",
    user_input="报错日志：IndexError: list index out of range",
    output_format="分成原因、修复建议、验证方式三部分",
    constraints="尽量少改代码，不要直接重写整个函数",
)

print(demo_instructions)

## 8. Demo 5：OpenAI 兼容接口的迁移方式

如果你使用的是 DeepSeek、Ollama 或其他 OpenAI 兼容接口，很多时候 Python 代码几乎不用改。

通常只需要调整两件事：

1. `base_url`
2. `model`

下面这个单元只打印客户端初始化方式，不真正发请求。

In [ ]:
examples = [
    {
        "name": "OpenAI",
        "base_url": "https://api.openai.com/v1",
        "model": "gpt-4.1-mini",
    },
    {
        "name": "DeepSeek",
        "base_url": "https://api.deepseek.com",
        "model": "deepseek-chat",
    },
    {
        "name": "Ollama(OpenAI-compatible)",
        "base_url": "http://localhost:11434/v1",
        "model": "qwen3:8b",
    },
]

for item in examples:
    print(json.dumps(item, ensure_ascii=False, indent=2))
    print("-" * 40)

## 9. 小结

这个 notebook 对应的核心结论可以收束成三句话：

- `System Prompt` 更适合放长期稳定规则
- `Instructions` 更适合放当前任务要求
- 用户原话往往只是素材来源，很多时候需要先整理，再变成真正的 instructions

如果要继续往下扩展，下一步最自然的方向是：

- 加一个多轮对话 demo，展示历史上下文如何污染 instructions
- 加一个工具调用 demo，展示 system 规则如何约束工具使用方式
- 加一个 notebook 小实验，对比“混写”和“分层写”的输出差异
